# Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
import datasets
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback
)
from transformers.trainer_utils import get_last_checkpoint
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import evaluate
import wandb
from datetime import datetime
import time
from tqdm.auto import tqdm
import sqlite3
import sqlparse
import _config

import os
import psutil
import GPUtil
import gc

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_API_KEY"] = _config.WANDB_API_KEY
os.environ["WANDB_PROJECT"] = _config.WANDB_PROJECT

ENABLE_THINKING = False

# Utils

In [ ]:
def get_vm_usage_metrics():
    # CPU usage
    cpu_load = psutil.cpu_percent(interval=1, percpu=True)
    for id, load in enumerate(cpu_load):
        print(f"CPU {id} load: {load:.2f}")
    # RAM usage
    ram = psutil.virtual_memory()
    print(f"RAM Total: {ram.total/(1024**3):.2f} GB, Used: {(ram.used)/(1024**3):.2f} GB")
    # GPU
    if torch.cuda.is_available():
        gpus = GPUtil.getGPUs()
        for gpu in gpus:
            print(f"GPU {gpu.id} ({gpu.name}) load: {gpu.load*100}%")
            print(f"GPU {gpu.id} ({gpu.name}) VRAM Total: {gpu.memoryTotal} MB, Used {gpu.memoryUsed} MB")
    # Disk 
    disk = psutil.disk_usage('/')
    print(f"Disk Total: {disk.total/(1024**3):.2f} GB, Used: {(disk.used)/(1024**3):.2f} GB")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Device: {device}')
get_vm_usage_metrics()

# Data

In [ ]:
data = pd.read_csv('preference_data.xlsx')

print(data.shape)
data.head()

In [ ]:
dataset = []
for id in range(data.shape[0]):
    dataset.append({
        'prompt': [{'role': 'user', 'content': data.loc[id, 'sql_prompt']}],
        'chosen': [{'role': 'assistant', 'content': data.loc[id, 'sql']}],
        'rejected': [{'role': 'assistant', 'content': data.loc[id, 'completion']}]
    })
dataset = datasets.Dataset.from_list(dataset)

split = dataset.train_test_split(test_size=0.1, seed=42)
ds_train = split['train']
ds_valid = split['test']

ds_train

# Models

In [ ]:
checkpoint = "Qwen/Qwen3-0.6B"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # change to bf16 if availiable
)

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    device_map="auto",
    attn_implementation=training_config["attn_implementation"],
    quantization_config=bnb_config,
    # dtype=torch.bfloat16 # not specifying on T4 to avoid unscaling errors
)

model.config.use_cache = False
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
model.enable_input_require_grads()

get_vm_usage_metrics()

# DPO

In [ ]:
torch.cuda.empty_cache()

# --- hyperparameters ---
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
RUN_NAME = f'dpo-qlora-{timestamp}'
OUTPUT_DIR = './dpo-output'
RESUME_TRAINING = False

PER_DEVICE_BATCH_SIZE = 2
effective_batch_size = 16
epochs = 1
learning_rate = 1e-5
lr_scheduler_type = "cosine"
weight_decay = 0.0
betas = (0.95, 0.999)
EPSILON = 1e-6  # safer than default 1e-8 for bf16/fp16 training
warmup_ratio = 0.1
lora_r = 16 * 4
lora_alpha = 64 * 4
lora_dropout = 0.01
optimizer = 'adamw'
seed = 42
NUM_EVALS_PER_RUN = 32  # target number of evals per run; guarantees at least 1

gradient_accumulation_steps = max(1, effective_batch_size // PER_DEVICE_BATCH_SIZE)

# --- compute steps BEFORE TrainingArguments so eval_steps is always valid ---
dataset_size = len(ds_train)
steps_per_epoch = max(1, dataset_size // (PER_DEVICE_BATCH_SIZE * gradient_accumulation_steps))
total_steps = steps_per_epoch * epochs
warmup_steps = int(total_steps * warmup_ratio)

# Spread evals evenly across training; clamp so eval_steps <= total_steps,
# guaranteeing at least one eval even with very large batches / small datasets
eval_steps = max(1, min(total_steps, total_steps // NUM_EVALS_PER_RUN))

# --- seed everything ---
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
transformers.set_seed(seed)

# --- W&B init ---
wandb.init(
    project=os.environ["WANDB_PROJECT"],
    name=RUN_NAME,
    # id=run_id,      # resume previous run if available
    # resume="allow", # allows resuming crashed run
)

# --- optimizer map ---
if training_config["fp16"]:
    optimizer_map = {
        "adam":   torch.optim.Adam,
        "adamw":  torch.optim.AdamW,
        "nadam":  torch.optim.NAdam,
        "adamax": torch.optim.Adamax,
    }
else:
    optimizer_map = {
        "adam":   bnb.optim.Adam8bit,
        "adamw":  bnb.optim.AdamW8bit,
        "nadam":  torch.optim.NAdam,
        "adamax": torch.optim.Adamax,
    }

training_args = DPOConfig(
    output_dir=OUTPUT_DIR,

    num_train_epochs=epochs,
    beta=0.1,

    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    lr_scheduler_type=lr_scheduler_type,
    weight_decay=weight_decay,
    warmup_ratio=warmup_ratio,
    max_grad_norm=1,

    bf16=training_config['bf16'],
    fp16=training_config['fp16'],
    bf16_full_eval=training_config['bf16'],
    fp16_full_eval=training_config['fp16'],
    tf32=training_config.get('tf32', False),

    use_liger_kernel=training_config['liger_kernel'],
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    save_strategy="steps",
    save_steps=eval_steps,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=eval_steps,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE * 2,
    eval_accumulation_steps=4,
    logging_strategy="steps",
    logging_steps=1,

    report_to=['wandb'],
    run_name=RUN_NAME,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    seed=seed,
    # generate_during_eval=True
)

# --- optimizer builder ---
def build_optimizer(model):
    optimizer_class = optimizer_map[optimizer]
    return optimizer_class(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
        betas=betas,
        eps=EPSILON,
    )

peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules='all-linear'
)
model.requires_grad_(False)                      # freeze base weights (precautionary)
model_peft = get_peft_model(model, peft_config)  # inject a LoRA adapter

trainer = DPOTrainer(
    processing_class=tokenizer,
    model=model_peft,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_valid,
    optimizers=(build_optimizer(model_peft), None),  # (optimizer, scheduler)
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# --- training setup summary ---
print("===== Training Setup Summary =====")
print(f"Num epochs:            {epochs}")
print(f"Effective batch size:  {effective_batch_size}")
print(f"Per-device batch size: {PER_DEVICE_BATCH_SIZE}")
print(f"Gradient accumulation: {gradient_accumulation_steps}")
print(f"LR scheduler:          {lr_scheduler_type}")
print(f"Optimizer:             {optimizer}")
print(f"Epsilon:               {EPSILON}")
print(f"Betas:                 {betas}")
print(f"Weight decay:          {weight_decay}")
print(f"DPO beta:              {training_args.beta}")
print(f"Dataset size:          {dataset_size}")
print(f"Steps per epoch:       {steps_per_epoch}")
print(f"Total training steps:  {total_steps}")
print(f"Warmup steps:          {warmup_steps}")
print(f"Eval steps:            {eval_steps}  (target {NUM_EVALS_PER_RUN} evals/run)")
print(f"Logging steps:         {training_args.logging_steps}")
print("===================================")
print(f"Start time: {datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}")

# --- training ---
last_checkpoint = None
if RESUME_TRAINING and os.path.isdir(OUTPUT_DIR):
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint is not None:
    print(f"Resuming training from checkpoint: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting fresh training run")
    trainer.train()

print(f"End time: {datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}")

# --- W&B logging of eval metrics ---
for log in trainer.state.log_history:
    if 'eval_loss' in log:
        wandb.log({
            "eval_loss":            log['eval_loss'],
            "eval_perplexity":      math.exp(log['eval_loss']),
            "step":                 log['step'],
            "learning_rate":        learning_rate,
            "lr_scheduler_type":    lr_scheduler_type,
            "weight_decay":         weight_decay,
            "betas":                betas,
            "warmup_ratio":         warmup_ratio,
            "effective_batch_size": effective_batch_size,
            "optimizer":            optimizer,
            "epsilon":              EPSILON,
            "dpo_beta":             training_args.beta,
            "seed":                 seed,
        })

wandb.finish()

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/best_model")

# Test

In [ ]:
OUTPUT_DIR = './dpo-output'
checkpoint = f"{OUTPUT_DIR}/checkpoint-391/"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint, dtype=torch.float16).to(device)
model.eval()

ds = datasets.load_dataset('gretelai/synthetic_text_to_sql', streaming=False)
ds_train, ds_test = ds['train'], ds['test']


def construct_message(prompt, context):
    return [
        {"role": "system", "content": f"The user asks a question. Your task is to generate the SQL query to answer that question. Return SQL query only. The context of the question is the following: '{context}'"},
        {"role": "user", "content": prompt}
    ]

def generate_model_response_batch(messages_list, enable_thinking=True, max_new_tokens=512):
    texts = [
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking
        )
        for messages in messages_list
    ]

    model_inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        padding_side='left'
    ).to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens
    )

    responses = []
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids):
        # Slice to get only generated part
        output_only_ids = output_ids[len(input_ids):].tolist()

        # Try to find `</think>` (id 151668)
        try:
            index = len(output_only_ids) - output_only_ids[::-1].index(151668)
        except ValueError:
            index = 0

        if enable_thinking:
            thinking_content = tokenizer.decode(
                output_only_ids[:index],
                skip_special_tokens=True
            ).strip("\n")
            content = tokenizer.decode(
                output_only_ids[index:],
                skip_special_tokens=True
            ).strip("\n")
        else:
            thinking_content = None
            content = tokenizer.decode(
                output_only_ids,
                skip_special_tokens=True
            ).strip("\n")

        responses.append({
            'thinking_content': thinking_content,
            'content': content
        })

    return responses


rouge = evaluate.load("rouge")

def normalize_sql(sql):
    return sqlparse.format(sql, reindent=True, keyword_case='upper').strip()

def compute_rouge(reference, prediction):
    result = rouge.compute(predictions=[prediction], references=[reference])
    return result['rougeL']

def evaluate_sql_response(reference, prediction, sql_context):
    # ROUGE-L
    rouge_score = compute_rouge(reference, prediction)
    
    # execution check
    try:
        conn = sqlite3.connect(":memory:")
        cursor = conn.cursor()
        
        cursor.executescript(sql_context)
        cursor.execute(reference)
        ref_result = cursor.fetchall()
        
        cursor.execute(prediction)
        model_result = cursor.fetchall()
        
        execution_match = ref_result == model_result
    except Exception:
        execution_match = False
    finally:
        conn.close()
    
    # final score
    if execution_match:
        final_score = 1.0
    else:
        final_score = 0.7 * rouge_score

    return {
        "rougeL": round(rouge_score, 4),
        "execution_match": execution_match,
        "final_score": final_score
    }

In [ ]:
BATCH_SIZE = 32
ENABLE_THINKING = False
MAX_NEW_TOKENS = 512


prompts = [ds_test[id]['sql_prompt'] for id in range(len(ds_test))]
contexts = [ds_test[id]['sql_context'] for id in range(len(ds_test))]

responses = []
print(f"Start time: {time.ctime(time.time())}")
for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
    batch_prompts = prompts[i : i + BATCH_SIZE]
    batch_contexts = contexts[i : i + BATCH_SIZE]

    messages_list = [
        construct_message(prompt=p, context=c)
        for p, c in zip(batch_prompts, batch_contexts)
    ]

    batch_responses = generate_model_response_batch(messages_list, enable_thinking=ENABLE_THINKING, max_new_tokens=MAX_NEW_TOKENS)

    responses.extend(batch_responses)

print(f"End time: {time.ctime(time.time())}")

In [ ]:
references = [ds_test[id]['sql'] for id in range(len(ds_test))]
predictions = [responses[id]['content'] for id in range(len(ds_test))]

scores = [
    evaluate_sql_response(
        reference=reference,
        prediction=prediction,
        sql_context=context
    )
    for reference, prediction, context in tqdm(zip(references, predictions, contexts), total=len(ds_test))
]

In [ ]:
print(f"Mean test set score: {np.mean([score['final_score'] for score in scores]):.3f}")